In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master('local[*]').appName('serious').getOrCreate()

26/02/22 19:57:34 WARN Utils: Your hostname, codespaces-473e1b resolves to a loopback address: 127.0.0.1; using 10.0.1.134 instead (on interface eth0)
26/02/22 19:57:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/02/22 19:57:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [3]:
df_green = df_green\
                .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime')\
                .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')
                
df_yellow = df_yellow\
                .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime')\
                .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')                

In [6]:
df_green.show()

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2|2019-12-18 15:52:30|2019-12-18 15:54:39|                 N|         1|         264|         264|              5|          0.0|        3.5|  0.5|    0.5|      0.01

In [7]:
df_yellow.show()

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1|2020-01-01 00:28:15|2020-01-01 00:33:03|              1|          1.2|         1|                 N|         238|         239|           1|        6.0|  3.0|    0.5|      1.47|         0.0|                  0.3|       11.2

In [17]:
df_green.registerTempTable('green')

In [37]:
df_green_revenue = spark.sql(
                        """
                        SELECT
                            date_trunc('hour', pickup_datetime) as hour,
                            PULocationID as zone,
                            
                            sum(total_amount) as amount,
                            count(*) as num_trips
                        FROM 
                            green
                        WHERE pickup_datetime >= '2020-01-01 00:00:00'
                        GROUP BY 1, 2
                        ORDER BY 1, 2
                        """
                    )


In [39]:
df_green_revenue.write.parquet('data/reports/revenue/green/')

In [40]:
df_yellow.registerTempTable('yellow')

In [42]:
df_yellow_revenue = spark.sql(
                        """
                        SELECT
                            date_trunc('hour', pickup_datetime) as hour,
                            PULocationID as zone,
                            
                            sum(total_amount) as amount,
                            count(*) as num_trips
                        FROM 
                            yellow
                        WHERE pickup_datetime >= '2020-01-01 00:00:00'
                        GROUP BY 1, 2
                        ORDER BY 1, 2
                        """
                    )

In [43]:
df_yellow_revenue.write.parquet('data/reports/revenue/yellow/')

In [46]:
df_green_prejoin = df_green_revenue\
                        .withColumnRenamed('amount', 'amount_green')\
                        .withColumnRenamed('num_trips', 'num_trips_green')
df_yellow_prejoin = df_yellow_revenue\
                        .withColumnRenamed('amount', 'amount_yellow')\
                        .withColumnRenamed('num_trips', 'num_trips_yellow')


df_joined = df_green_prejoin.join(df_yellow_prejoin, on=['hour', 'zone'], how='outer')

In [47]:
df_joined.show()

+-------------------+----+-----------------+---------------+------------------+----------------+
|               hour|zone|     amount_green|num_trips_green|     amount_yellow|num_trips_yellow|
+-------------------+----+-----------------+---------------+------------------+----------------+
|2020-01-01 00:00:00|   3|             null|           null|              25.0|               1|
|2020-01-01 01:00:00|  17|598.1499999999999|             18|            464.51|              18|
|2020-01-01 01:00:00| 107|             null|           null| 9994.479999999992|             583|
|2020-01-01 01:00:00| 162|             null|           null| 5736.690000000017|             298|
|2020-01-01 02:00:00| 234|             null|           null| 6759.990000000024|             370|
|2020-01-01 03:00:00| 170|             null|           null| 4632.000000000015|             257|
|2020-01-01 04:00:00|  22|             null|           null|             12.96|               1|
|2020-01-01 06:00:00| 255|    

In [48]:
df_zones = spark.read.parquet('data/pq/zones/*')
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [49]:
df_joined = df_joined.join(df_zones, df_joined.zone == df_zones.LocationID)

In [54]:
df_joined.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')